# Silver → Gold: dim_customer

**Propósito:** Leer `dbo.dim_sap_customers` del lakehouse Silver y escribir como `dbo.dim_customer` en Gold. Se renombra para que el modelo semántico tenga nomenclatura limpia.

**Transformaciones aplicadas:**
- Drop de `name_groups` (duplicado de `name` con diferencias mínimas)
- Drop de columnas siempre nulas en origen: `category`, `customer_sales_track`, `exclude_transfer_sales_track`, `center_type`
- Drop de `tax_number` (dato sensible, no necesario para analítica)
- Columna de auditoría `_gold_load_ts`

**Idempotencia:** `overwrite` + `overwriteSchema=true`.

In [1]:
%run ./config

StatementMeta(, e5c9fb67-409f-48f0-82e4-8e20e39aa868, 3, Finished, Available, Finished, True)

Config cargada → Bronze: lakehouse_poc | Silver: silver_lakehouse_poc | Gold: gold_lakehouse_poc


In [2]:


SILVER_TABLE = f"{DEFAULT_SCHEMA}.dim_sap_customers"
GOLD_TABLE   = f"{DEFAULT_SCHEMA}.dim_customer"

StatementMeta(, e5c9fb67-409f-48f0-82e4-8e20e39aa868, 4, Finished, Available, Finished, False)

In [3]:
from pyspark.sql import functions as F
from datetime import datetime

df_silver = spark.read.table(f"`{SILVER_LAKEHOUSE}`.{SILVER_TABLE}")

print(f"Filas leídas desde Silver: {df_silver.count()}")
df_silver.printSchema()

StatementMeta(, e5c9fb67-409f-48f0-82e4-8e20e39aa868, 5, Finished, Available, Finished, False)

Filas leídas desde Silver: 104430
root
 |-- customer_id: string (nullable = true)
 |-- country_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- region_id: string (nullable = true)
 |-- street: string (nullable = true)
 |-- customer_blocked: boolean (nullable = true)
 |-- tax_number: string (nullable = true)
 |-- category: string (nullable = true)
 |-- center_type: string (nullable = true)
 |-- customer_sales_track: string (nullable = true)
 |-- exclude_transfer_sales_track: string (nullable = true)
 |-- region: string (nullable = true)
 |-- country: string (nullable = true)
 |-- name_groups: string (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- _silver_load_ts: timestamp (nullable = true)



In [4]:
# ─── TRANSFORMACIONES ─────────────────────────────────────────────────────────
COLS_TO_DROP = [
    "name_groups",                      # duplicado de name
    "category",                         # nulo en origen
    "customer_sales_track",             # nulo en origen
    "exclude_transfer_sales_track",     # nulo en origen
    "center_type",                      # valor constante 'TG', sin aporte analítico
    "tax_number",                       # dato sensible, no necesario para analítica
    "_silver_load_ts",
]

existing_cols = df_silver.columns
cols_to_drop  = [c for c in COLS_TO_DROP if c in existing_cols]

df_gold = (
    df_silver
    .drop(*cols_to_drop)
    .withColumn("_gold_load_ts", F.lit(datetime.utcnow().isoformat()).cast("timestamp"))
)

StatementMeta(, e5c9fb67-409f-48f0-82e4-8e20e39aa868, 6, Finished, Available, Finished, False)

In [5]:
# ─── VALIDACIÓN ───────────────────────────────────────────────────────────────
row_count    = df_gold.count()
null_ids     = df_gold.filter(F.col("customer_id").isNull()).count()
dup_ids      = df_gold.groupBy("customer_id").count().filter(F.col("count") > 1).count()
active_count = df_gold.filter(F.col("is_active") == True).count()

print(f"Filas a escribir en Gold  : {row_count}")
print(f"customer_id nulos         : {null_ids}")
print(f"customer_id duplicados    : {dup_ids}")
print(f"Clientes activos          : {active_count} / {row_count}")

assert null_ids == 0, "ERROR: hay customer_id nulos en dim_customer Gold"
assert dup_ids  == 0, "ERROR: hay customer_id duplicados en dim_customer Gold"

df_gold.show(3)

StatementMeta(, e5c9fb67-409f-48f0-82e4-8e20e39aa868, 7, Finished, Available, Finished, False)

Filas a escribir en Gold  : 104430
customer_id nulos         : 0
customer_id duplicados    : 0
Clientes activos          : 0 / 104430
+-----------+----------+--------------------+--------------------+-----------+---------+--------------------+----------------+---------+-------+---------+--------------------+
|customer_id|country_id|                name|                city|postal_code|region_id|              street|customer_blocked|   region|country|is_active|       _gold_load_ts|
+-----------+----------+--------------------+--------------------+-----------+---------+--------------------+----------------+---------+-------+---------+--------------------+
|      29509|        ES|CARREFOUR GRAN VI...|HOSPITALET DE LLO...|      08908|       08|CAL. GV CORTS CAT...|            NULL|Barcelona| España|     NULL|2026-04-22 11:23:...|
|      29511|        ES|   CARREFOUR BARBERA|  BARBERÀ DEL VALLÈS|      08210|       08|CAL. CAR BARCELON...|            NULL|Barcelona| España|     NULL|2026-04-

In [6]:
# ─── ESCRITURA IDEMPOTENTE EN GOLD ────────────────────────────────────────────
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"`{GOLD_LAKEHOUSE}`.{GOLD_TABLE}")
)

print(f"✓ Tabla {GOLD_LAKEHOUSE}.{GOLD_TABLE} escrita correctamente con {row_count} filas.")

StatementMeta(, e5c9fb67-409f-48f0-82e4-8e20e39aa868, 8, Finished, Available, Finished, True)

✓ Tabla gold_lakehouse_poc.dbo.dim_customer escrita correctamente con 104430 filas.
